# Phase 1 walkthrough: GSE294180 liver RNA-seq

Read-only tour of the outputs of `scripts/01`–`06`. Interpretation lives in `reports/`; this notebook shows the underlying tables.

Re-generate with `python scripts/07_build_walkthrough_notebook.py`.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
T = ROOT / 'results' / 'tables'
pd.set_option('display.width', 160)

## 1. Samples (from GEO records)

In [2]:
meta = pd.read_csv(ROOT / 'metadata' / 'sample_metadata.csv')
meta[['geo_accession', 'counts_column', 'group', 'diet', 'microplastic',
      'library_name', 'geo_batch']]

,geo_accession,counts_column,group,diet,microplastic,library_name,geo_batch
0,GSM8899917,CD_1_S39,CD+Veh,CD,Veh,S39,B1
1,GSM8899918,CD_2_S40,CD+Veh,CD,Veh,S40,B1
2,GSM8899919,CD_3_S41,CD+Veh,CD,Veh,S41,B1
3,GSM8899920,CD_4_S42,CD+Veh,CD,Veh,S42,B1
4,GSM8899926,CD_MP_1_S48,CD+PE,CD,PE,S48,B1
5,GSM8899927,CD_MP_2_S49,CD+PE,CD,PE,S49,B1
6,GSM8899928,CD_MP_3_S50,CD+PE,CD,PE,S50,B1
7,GSM8899929,CD_MP_4_S51,CD+PE,CD,PE,S51,B1
8,GSM8899930,CD_MP_5_S52,CD+PE,CD,PE,S52,B1
9,GSM8899921,MD_1_S43,MD+Veh,MD,Veh,S43,B1


In [3]:
meta.groupby(['diet', 'microplastic']).size().unstack()

microplastic,PE,Veh
diet,,
CD,5,4
MD,5,5


## 2. Audit of the deposited count file

In [4]:
audit = json.loads((T / 'audit_summary.json').read_text())
pd.Series(audit['integrity'])

n_genes                                                                        55836
n_samples                                                                         19
dtypes                                                                       [int64]
gene_id_formats                        {'symbol': 55359, 'ensembl_unversioned': 477}
n_duplicate_gene_ids                                                               0
n_duplicate_ids_after_version_strip                                                0
n_duplicate_count_profiles                                                      3833
n_missing_values                                                                   0
n_infinite_values                                                                  0
n_negative_values                                                                  0
min_value                                                                        0.0
max_value                                                        

In [5]:
pd.Series(audit['value_type_evidence'])

column_sum_min                                    10398820.0
column_sum_max                                    15456729.0
columns_sum_to_1e6                                     False
non_integer_fraction                                     0.0
min_nonzero_value                                        1.0
fraction_nonzero_below_1                                 0.0
fraction_values_integer_multiple_of_column_min           1.0
n_fractional_values                                        0
fractional_part_quartiles                                 []
dtype: object

In [6]:
pd.read_csv(T / 'counts_preview.csv').iloc[:5, :6]

,Unnamed: 0,CD_MP_1_S48,CD_MP_2_S49,CD_MP_3_S50,CD_MP_4_S51,CD_MP_5_S52
0,0610005C13Rik,1975,1512,2048,1908,1755
1,0610006L08Rik,0,0,0,0,0
2,0610009E02Rik,8,4,4,11,7
3,0610009L18Rik,4,0,2,0,5
4,0610010K14Rik,81,78,64,64,64


In [7]:
pd.read_csv(T / 'per_sample_audit.csv', index_col=0)[
    ['group', 'total_counts', 'n_genes_ge_10', 'fraction_counts_chrM',
     'top_gene_symbol']]

,group,total_counts,n_genes_ge_10,fraction_counts_chrM,top_gene_symbol
counts_column,,,,,
CD_MP_1_S48,CD+PE,12654401,12795,0.118198,Alb
CD_MP_2_S49,CD+PE,10837229,12698,0.106767,Alb
CD_MP_3_S50,CD+PE,11788019,12429,0.119096,Alb
CD_MP_4_S51,CD+PE,10701996,12190,0.115763,Alb
CD_MP_5_S52,CD+PE,10398820,12176,0.105325,Alb
CD_1_S39,CD+Veh,15391969,13158,0.074026,Alb
CD_2_S40,CD+Veh,13940970,13111,0.091759,Alb
CD_3_S41,CD+Veh,15456729,12882,0.088764,Alb
CD_4_S42,CD+Veh,14787494,13000,0.086790,Alb


In [8]:
pd.read_csv(T / 'sex_marker_counts.csv', index_col=0)

,Xist,Ddx3y,Eif2s3y,Kdm5d,Uty,group
counts_column,,,,,,
CD_MP_1_S48,1,710,589,239,162,CD+PE
CD_MP_2_S49,0,765,490,184,135,CD+PE
CD_MP_3_S50,1,618,504,220,145,CD+PE
CD_MP_4_S51,0,605,487,173,144,CD+PE
CD_MP_5_S52,0,607,430,155,98,CD+PE
CD_1_S39,2,1073,718,246,191,CD+Veh
CD_2_S40,0,738,723,234,171,CD+Veh
CD_3_S41,0,730,610,226,164,CD+Veh
CD_4_S42,2,901,736,256,205,CD+Veh


## 3. Preprocessing

In [9]:
json.loads((T / 'filter_summary.json').read_text())

{'rule': '>= 10 counts in >= 4 samples',
 'n_genes_input': 55836,
 'n_genes_kept': 13777,
 'n_genes_removed': 42059,
 'fraction_of_total_counts_kept': 0.9986122310085155,
 'kept_by_gene_type': {'protein_coding': 12466,
  'lncRNA': 839,
  'processed_pseudogene': 240,
  'TEC': 130,
  'unprocessed_pseudogene': 39,
  'lncRNA;transcribed_unprocessed_pseudogene': 17,
  'transcribed_processed_pseudogene': 10,
  'transcribed_unprocessed_pseudogene': 9},
 'size_factor_range': [0.7819906593593199, 1.2633724354560638]}

In [10]:
pd.read_csv(T / 'size_factors.csv', index_col=0).round(3)

,library_size_all_genes,library_size_filtered_genes,size_factor,library_size_relative_to_mean,group
counts_column,,,,,
CD_1_S39,15391969,15377119,1.263,1.178,CD+Veh
CD_2_S40,13940970,13925581,1.087,1.067,CD+Veh
CD_3_S41,15456729,15443885,1.166,1.184,CD+Veh
CD_4_S42,14787494,14773736,1.152,1.132,CD+Veh
CD_MP_1_S48,12654401,12636829,0.944,0.968,CD+PE
CD_MP_2_S49,10837229,10825452,0.880,0.830,CD+PE
CD_MP_3_S50,11788019,11778136,0.885,0.903,CD+PE
CD_MP_4_S51,10701996,10693019,0.782,0.819,CD+PE
CD_MP_5_S52,10398820,10390197,0.789,0.796,CD+PE


## 4. Descriptive EDA

Figures (PNG and PDF; each one is explained in `reports/eda_report.md`):

- [fig01_library_sizes](../figures/eda/fig01_library_sizes.png)
- [fig02_count_distributions](../figures/eda/fig02_count_distributions.png)
- [fig03_sample_boxplots_before_after_normalisation](../figures/eda/fig03_sample_boxplots_before_after_normalisation.png)
- [fig04_expression_dominance](../figures/eda/fig04_expression_dominance.png)
- [fig05_mean_variance](../figures/eda/fig05_mean_variance.png)
- [fig06_sample_qc_by_group](../figures/eda/fig06_sample_qc_by_group.png)
- [fig07_selected_genes_by_group](../figures/eda/fig07_selected_genes_by_group.png)
- [fig08_variance_share_by_factor](../figures/eda/fig08_variance_share_by_factor.png)
- [fig09_tissue_marker_screen](../figures/eda/fig09_tissue_marker_screen.png)

In [11]:
pd.read_csv(T / 'variance_share_by_factor.csv')

,factor,median_eta2,median_eta2_shuffled,n_genes_eta2_gt_0.5,n_genes_eta2_gt_0.5_shuffled,n_genes_considered
0,Diet (CD vs MD),0.105382,0.025026,984,3,13164
1,Microplastic (Veh vs PE),0.052843,0.039535,252,11,13164
2,Group (4 levels),0.302541,0.147290,2871,134,13164


In [12]:
pd.read_csv(T / 'tissue_marker_cpm.csv', index_col=0).round(1)

,skeletal_muscle,adipocyte,erythrocyte,pancreas_exocrine,cholangiocyte,kupffer_macrophage,stellate_fibrogenic,group
counts_column,,,,,,,,
CD_1_S39,1.5,0.0,134.6,1.0,49.4,134.0,169.2,CD+Veh
CD_2_S40,58.0,1.2,178.7,1.9,42.2,74.0,155.7,CD+Veh
CD_3_S41,2.0,0.1,162.7,0.8,25.4,130.2,104.9,CD+Veh
CD_4_S42,1.8,0.1,184.6,0.3,43.3,131.4,137.6,CD+Veh
CD_MP_1_S48,539.1,0.5,120.6,0.6,29.4,87.3,152.1,CD+PE
CD_MP_2_S49,10.4,0.5,405.1,4.0,28.4,572.8,168.1,CD+PE
CD_MP_3_S50,4.1,0.0,98.9,0.3,35.6,88.6,127.9,CD+PE
CD_MP_4_S51,1.8,0.0,109.6,2.6,30.3,76.3,139.3,CD+PE
CD_MP_5_S52,1.5,0.1,106.1,0.2,43.0,59.5,157.3,CD+PE


## 5. PCA and clustering

Figures (PNG and PDF; each one is explained in `reports/eda_report.md`):

- [fig10_pca_primary](../figures/pca_clustering/fig10_pca_primary.png)
- [fig11_pca_top_loadings](../figures/pca_clustering/fig11_pca_top_loadings.png)
- [fig12_pca_all_filtered_genes](../figures/pca_clustering/fig12_pca_all_filtered_genes.png)
- [fig13_pca_muscle_genes_removed](../figures/pca_clustering/fig13_pca_muscle_genes_removed.png)
- [fig14_sample_correlation_heatmap](../figures/pca_clustering/fig14_sample_correlation_heatmap.png)
- [fig15_dendrograms_sensitivity](../figures/pca_clustering/fig15_dendrograms_sensitivity.png)

In [13]:
pd.read_csv(T / 'pca_variance_explained.csv').iloc[:, :7]

,feature_set,n_genes,PC1_pct,PC2_pct,PC3_pct,PC4_pct,PC5_pct
0,top500_log2norm,500,30.33,17.04,8.47,6.24,4.84
1,top1000_log2norm,1000,26.33,17.01,8.39,6.85,4.98
2,top2000_log2norm,2000,23.46,16.88,8.63,7.05,5.53
3,allfiltered_log2norm,13777,20.71,16.22,8.48,7.17,5.99
4,top500_log2cpm,500,34.39,24.91,8.52,6.55,4.76
5,top500_log2norm_muscle_genes_removed,500,20.84,17.94,9.03,6.84,5.82
6,top500_log2norm_sample_MD_1_removed,500,20.79,19.47,10.21,7.37,5.59


In [14]:
assoc = pd.read_csv(T / 'pca_pc_associations.csv')
assoc[assoc['PC'].isin(['PC1', 'PC2'])]

,feature_set,PC,eta2_diet,eta2_microplastic,eta2_group,spearman_r_library_size,spearman_r_library_index,spearman_r_muscle_marker_cpm
0,top500_log2norm,PC1,0.084,0.183,0.371,0.491,-0.370,0.791
1,top500_log2norm,PC2,0.692,0.230,0.975,0.277,-0.721,-0.098
5,top1000_log2norm,PC1,0.138,0.158,0.381,0.404,-0.198,0.795
6,top1000_log2norm,PC2,0.683,0.228,0.958,0.321,-0.740,-0.012
10,top2000_log2norm,PC1,0.259,0.110,0.437,0.374,-0.142,0.754
11,top2000_log2norm,PC2,0.574,0.259,0.879,0.404,-0.814,0.123
15,allfiltered_log2norm,PC1,0.545,0.027,0.622,0.225,0.181,0.593
16,allfiltered_log2norm,PC2,0.285,0.377,0.710,0.528,-0.854,0.396
20,top500_log2cpm,PC1,0.289,0.059,0.440,0.232,0.104,0.686
21,top500_log2cpm,PC2,0.542,0.270,0.859,-0.393,0.818,-0.098


In [15]:
agree = pd.read_csv(T / 'clustering_label_agreement.csv')
agree.pivot_table(index=['clustering', 'k'], columns='label',
                  values='adjusted_rand_index').round(2)

label                                                 diet  group  microplastic
clustering                                         k                           
1 - Pearson, average (primary)                     2 -0.01   0.05          0.03
                                                   3  0.02   0.07          0.07
                                                   4  0.65   0.46          0.03
                                                   5  0.58   0.39          0.00
                                                   6  0.56   0.35         -0.02
1 - Pearson, average, muscle-tracking genes rem... 2  1.00   0.46         -0.06
                                                   3  0.75   0.49          0.04
                                                   4  0.65   0.46          0.03
                                                   5  0.58   0.39          0.00
                                                   6  0.38   0.59          0.19
1 - Spearman, average                              2 -0.01   0.05          0.03
                                                   3  0.10   0.32          0.34
                                                   4  0.14   0.34          0.37
                                                   5  0.46   0.65          0.22
                                                   6  0.42   0.58          0.18
Euclidean, Ward                                    2 -0.01   0.05          0.03
                                                   3  0.62   0.42         -0.00
                                                   4  0.42   0.61          0.18
                                                   5  0.46   0.65          0.22
                                                   6  0.43   0.60          0.19

## 6. Where to read the interpretation

- `reports/dataset_audit_report.md`: data issues
- `reports/eda_report.md`: every figure explained
- `reports/preliminary_observations.md`: candidate questions for Phase 2